# CoSQA E5-base-v2 baseline

This notebook is the ordered entry point for the AZH-514 baseline. It evaluates `intfloat/e5-base-v2` on the pinned `CoIR-Retrieval/cosqa` test qrels using the complete code corpus and the official COIR evaluator at `nDCG@10`.

The default execution mode is a small **real-model smoke run**. Smoke output proves wiring only and is not benchmark evidence. Set `E5_BASELINE_MODE=benchmark` before execution for the complete declared test split and corpus.

Reference: [CoIR: A Comprehensive Benchmark for Code Information Retrieval Models](https://arxiv.org/pdf/2407.02883). The paper reports the CosQA task as natural-language queries over a corpus of Python code snippets and uses `NDCG@10`.

In [ ]:
# Install code-retrieval/requirements.txt once before running this notebook if needed.
# In Colab, uncomment the next line and restart the runtime if pip updates torch/numpy.
# %pip install -r ../requirements.txt

from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import sys
import time

workspace_root = Path.cwd()
while workspace_root != workspace_root.parent and not (workspace_root / 'code-retrieval' / 'src').exists():
    workspace_root = workspace_root.parent
project_dir = workspace_root / 'code-retrieval'
if not (project_dir / 'src').exists():
    project_dir = Path.cwd()
    workspace_root = project_dir.parent

os.chdir(project_dir)
sys.path.insert(0, str(project_dir / 'src'))

from e5_baseline import (
    BaselineConfig,
    E5Encoder,
    build_result,
    cache_paths,
    environment_metadata,
    evaluate_ndcg_at_10,
    expected_cache_metadata,
    load_cosqa,
    load_valid_embedding_cache,
    load_valid_json_cache,
    package_versions,
    rank_with_faiss,
    run_identity,
    save_embedding_cache,
    save_json_cache,
    select_run_data,
    set_seed,
    sha256_text,
    write_result_artifacts,
)

print('Project directory:', project_dir.resolve())
print('Python:', sys.version.split()[0])

## 1. Environment and explicit controls

All controls that can affect comparable runs are declared here. The dataset and model revisions are immutable Hugging Face commit SHAs observed on 2026-09-18. The COIR package is pinned in `requirements.txt` and checked below.

In [ ]:
RUN_MODE = os.environ.get('E5_BASELINE_MODE', 'smoke').strip().lower()
if RUN_MODE not in {'smoke', 'benchmark'}:
    raise ValueError('E5_BASELINE_MODE must be smoke or benchmark')

batch_size = int(os.environ.get('E5_BASELINE_BATCH_SIZE', '32' if RUN_MODE == 'smoke' else '64'))
torch_threads = int(os.environ.get('E5_BASELINE_TORCH_THREADS', '0'))
if torch_threads > 0:
    import torch
    torch.set_num_threads(torch_threads)

config = BaselineConfig(
    run_mode=RUN_MODE,
    batch_size=batch_size,
    cache_dir='artifacts/e5_baseline/cache',
    artifact_dir='artifacts/e5_baseline',
)
set_seed(config.seed)

print(json.dumps(config.as_dict(), indent=2, sort_keys=True))
print('Installed packages:')
print(json.dumps(package_versions(['coir-eval', 'datasets', 'faiss-cpu', 'numpy', 'pytrec-eval-terrier', 'sentence-transformers', 'torch', 'transformers']), indent=2, sort_keys=True))
print('Hardware/runtime:')
print(json.dumps(environment_metadata(config, repo_root=workspace_root), indent=2, sort_keys=True, default=str))

## 2. Load and inspect the real CosQA schema

The loader reads the separate `corpus`, `queries`, and `default/test` configurations from the same pinned dataset revision. It adapts only after checking the observed columns and preserves source IDs. Titles are intentionally excluded to match the paper-compatible COIR text-only path.

In [ ]:
data = load_cosqa(config)
print(json.dumps(data.schema, indent=2, sort_keys=True, default=str))
print({
    'corpus_count': len(data.corpus),
    'query_count_with_test_qrels': len(data.queries),
    'qrels_query_count': len(data.qrels),
    'qrels_judgment_count': sum(len(rels) for rels in data.qrels.values()),
})
print('Corpus example:', next(iter(data.corpus.items())))
print('Query example:', next(iter(data.queries.items())))
print('Qrels example:', next(iter(data.qrels.items())))

## 3. Select evidence level

Smoke mode keeps the first few labeled queries and a small corpus, then adds each selected query’s judged documents so the evaluator receives a valid non-empty ranking problem. Benchmark mode uses every test-qrels query and every corpus document.

In [ ]:
run_data = select_run_data(data, config)
if config.run_mode == 'benchmark' and (len(run_data.queries) != len(data.queries) or len(run_data.corpus) != len(data.corpus)):
    raise RuntimeError('benchmark mode must use the complete declared CosQA test queries and corpus')

print(json.dumps({
    'run_mode': config.run_mode,
    'query_count': len(run_data.queries),
    'corpus_count': len(run_data.corpus),
    'qrels_query_count': len(run_data.qrels),
    'qrels_judgment_count': sum(len(rels) for rels in run_data.qrels.values()),
    'exclusions': run_data.exclusions,
}, indent=2))

## 4. Run identity and cache paths

Corpus embeddings, query embeddings, and rankings are cached separately. A cache is accepted only when its metadata matches the full configuration, code version, repository commit, and ordered source-ID fingerprint.

In [ ]:
notebook_path = project_dir / 'notebooks' / 'e5_baseline_experiment.ipynb'
notebook_sha256 = hashlib.sha256(notebook_path.read_bytes()).hexdigest() if notebook_path.exists() else None
identity = run_identity(config, repo_root=workspace_root)
paths = cache_paths(config, identity)
print('Run identity:', identity)
print('Notebook SHA-256:', notebook_sha256)
print(json.dumps({name: str(path) for name, path in paths.items()}, indent=2))

## 5. Encode the corpus with E5

E5 receives `passage: ` for corpus text and `query: ` for query text. Sentence Transformers performs the model’s documented pooling; embeddings are normalized and truncated to 512 tokens. No title, answer, label, or target-document information is passed to the model.

In [ ]:
encoder = E5Encoder(config)
corpus_ids = list(run_data.corpus)
corpus_texts = [run_data.corpus[doc_id]['text'] for doc_id in corpus_ids]
corpus_metadata = expected_cache_metadata(
    config, identity=identity, kind='corpus_embeddings', ids=corpus_ids, repo_root=workspace_root
)
corpus_embeddings = load_valid_embedding_cache(paths['corpus_embeddings'], paths['corpus_metadata'], corpus_metadata)
if corpus_embeddings is None:
    corpus_started = time.perf_counter()
    corpus_embeddings = encoder.encode_corpus(corpus_texts)
    corpus_seconds = time.perf_counter() - corpus_started
    save_embedding_cache(paths['corpus_embeddings'], paths['corpus_metadata'], corpus_embeddings, corpus_metadata)
else:
    corpus_seconds = 0.0
print('Corpus embeddings:', corpus_embeddings.shape, 'seconds:', round(corpus_seconds, 3))

## 6. Encode evaluation queries

The original query text remains the evaluation query. Query expansion is intentionally absent from this baseline.

In [ ]:
query_ids = list(run_data.queries)
query_texts = [run_data.queries[query_id] for query_id in query_ids]
query_metadata = expected_cache_metadata(
    config, identity=identity, kind='query_embeddings', ids=query_ids, repo_root=workspace_root
)
query_embeddings = load_valid_embedding_cache(paths['query_embeddings'], paths['query_metadata'], query_metadata)
if query_embeddings is None:
    query_started = time.perf_counter()
    query_embeddings = encoder.encode_queries(query_texts)
    query_seconds = time.perf_counter() - query_started
    save_embedding_cache(paths['query_embeddings'], paths['query_metadata'], query_embeddings, query_metadata)
else:
    query_seconds = 0.0
print('Query embeddings:', query_embeddings.shape, 'seconds:', round(query_seconds, 3))

## 7. Exact first-stage retrieval

The ranking uses an exact Faiss `IndexFlatIP` over normalized embeddings, equivalent to cosine similarity. The re-ranker and HyDE are absent, and rankings cannot introduce documents outside the shared corpus.

In [ ]:
ranking_ids = query_ids + ['__corpus__'] + corpus_ids
ranking_metadata = expected_cache_metadata(
    config, identity=identity, kind='rankings', ids=ranking_ids, repo_root=workspace_root
)
rankings = load_valid_json_cache(paths['rankings'], paths['rankings_metadata'], ranking_metadata)
if rankings is None:
    ranking_started = time.perf_counter()
    rankings = rank_with_faiss(
        query_embeddings, corpus_embeddings, query_ids, corpus_ids, top_k=config.candidate_depth
    )
    ranking_seconds = time.perf_counter() - ranking_started
    save_json_cache(paths['rankings'], paths['rankings_metadata'], rankings, ranking_metadata)
else:
    ranking_seconds = 0.0
print('Ranking queries:', len(rankings))
print('First ranking:', next(iter(rankings.items())))

## 8. Official COIR evaluation

COIR’s `EvaluateRetrieval` implementation uses the project’s `pytrec-eval-terrier` dependency and returns `NDCG@10` along with secondary metrics. The aggregate is measured from the current rankings and qrels; it is never manually entered.

In [ ]:
evaluation_started = time.perf_counter()
metric = evaluate_ndcg_at_10(run_data.qrels, rankings, cutoff=10)
evaluation_seconds = time.perf_counter() - evaluation_started
print(json.dumps(metric, indent=2, sort_keys=True))

## 9. Persist result and provenance

A smoke result is explicitly marked `smoke` and `benchmark_evidence: false`. Only a complete benchmark-mode run over the declared query/corpus population is marked `benchmark`.

In [ ]:
environment = environment_metadata(config, repo_root=workspace_root)
result = build_result(
    config,
    data,
    run_data,
    metric,
    identity=identity,
    environment=environment,
    timings={
        'corpus_encoding': corpus_seconds,
        'query_encoding': query_seconds,
        'ranking': ranking_seconds,
        'evaluation': evaluation_seconds,
    },
    repo_root=workspace_root,
)
result['notebook_sha256'] = notebook_sha256
result['cache_paths'] = {name: str(path) for name, path in paths.items()}
artifact_paths = write_result_artifacts(
    config,
    result,
    metadata={
        'result': result,
        'cache_metadata': {
            'corpus': corpus_metadata,
            'queries': query_metadata,
            'rankings': ranking_metadata,
        },
    },
)
print(json.dumps({
    'status': result['status'],
    'benchmark_evidence': result['benchmark_evidence'],
    'nDCG@10': result['ndcg_at_10'],
    'query_count': result['query_count'],
    'corpus_count': result['corpus_count'],
    'artifact_paths': artifact_paths,
}, indent=2))

## Interpretation boundary

This issue delivers the E5 baseline only. Smoke metrics are useful for checking data/model/evaluator wiring but must not be interpreted as benchmark performance. Later HyDE, re-ranking, and combined experiments must reuse the same declared data/evaluator controls and compare against a complete `benchmark`-status E5 result.